---
## Task 1 — Total Goals per Match

### Analytic question formulation
Is the average number of **total goals scored per match** (both teams combined) at the FIFA
World Cup 2026 significantly different from **2.5 goals per match**, a commonly cited long-run
historical FIFA World Cup average?

### Data wrangling
Fetch the `Matches` sheet from the clean dataset and derive `TotalGoals` (Score 1 + Score 2).
The **population** here is every match played in the tournament.


In [9]:
import pandas as pd
import numpy as np
import re
from scipy import stats
CLEAN_PATH = "../World_Cup_2026_clean.xlsx"

matches_t1 = pd.read_excel(CLEAN_PATH, sheet_name="Matches")
matches_t1["TotalGoals"] = matches_t1["Score 1"] + matches_t1["Score 2"]
population_goals = matches_t1["TotalGoals"].dropna()

print("Population size (all matches):", len(population_goals))
print("Population mean goals/match:", round(population_goals.mean(), 3))

Population size (all matches): 103
Population mean goals/match: 2.893


### Data preparation and sampling
**Population:** all 103 completed matches of the tournament.
**Sample:** a **simple random sample of n = 30 matches**, drawn without replacement using
`pandas.Series.sample()` with a fixed random seed for reproducibility.


In [5]:
sample_goals = population_goals.sample(n=30, random_state=42)
print("Sample size:", len(sample_goals))

Sample size: 30


### Descriptive statistics

In [10]:
print(sample_goals.describe())
print("Skewness:", round(stats.skew(sample_goals), 3))

count    30.000000
mean      2.400000
std       1.566899
min       0.000000
25%       1.000000
50%       2.500000
75%       3.000000
max       6.000000
Name: TotalGoals, dtype: float64
Skewness: 0.243


### Inferential statistics — Confidence interval
95% confidence interval for the true mean number of goals per match, based on the sample
(using the t-distribution since the population standard deviation is unknown).


In [11]:
mean = sample_goals.mean()
sem = stats.sem(sample_goals)
ci = stats.t.interval(0.95, df=len(sample_goals) - 1, loc=mean, scale=sem)
print(f"Sample mean: {mean:.3f}, SEM: {sem:.3f}")
print(f"95% CI for mean total goals per match: ({ci[0]:.3f}, {ci[1]:.3f})")

Sample mean: 2.400, SEM: 0.286
95% CI for mean total goals per match: (1.815, 2.985)


### Inferential statistics — One-sample t-Test
H₀: μ = 2.5 goals/match  vs.  H₁: μ ≠ 2.5 goals/match


In [12]:
t_stat, p_val = stats.ttest_1samp(sample_goals, popmean=2.5)
print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.4f}")
alpha = 0.05
print("Conclusion:", "Reject H0" if p_val < alpha else "Fail to reject H0",
      "at the 5% significance level.")

t-statistic = -0.350, p-value = 0.7292
Conclusion: Fail to reject H0 at the 5% significance level.


**Interpretation:** With p = 0.729 (> 0.05), there is not enough evidence to conclude the
2026 World Cup's average goals-per-match differs from the historical benchmark of 2.5. The 95%
CI (1.82, 2.99) comfortably contains 2.5, consistent with this conclusion.
